# 玻璃棒：清洗、过滤与去重对照
只读取已完成的实验表，不重跑模型。完整原文、各分支输出、删除依据及HTML抽取对照都在最后一格。实验算子：`ops/cleaning_trials.py`；demiflow编排：`try_cleaning.py`。

In [ ]:
from pathlib import Path
import sys, json
from functools import partial
from IPython.display import HTML, display
from demiflow.standalone import local_data
ROOT=Path('/yzp/zhaozy/yangzepeng/0905/demiwtg')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from curation.notebook_image_preview import show as render
RUN=ROOT/'state/curation/cleaning_glass_v4'
data=local_data()
show=partial(render, run=RUN, dataset=ROOT/'datasets/demiwtg')
# 用文档ID前缀选一篇查看；空字符串显示全部14篇。
DOC=''

## 1．原始采集文本
14篇原文件已与datasets中的字节哈希核对；这里保留其完整内容。

In [ ]:
raw_documents=data.read_records(RUN/'raw_documents.jsonl').map(lambda r:r['value']).filter(lambda r:r['doc_id'].startswith(DOC))
show(raw_documents,columns=['doc_id','title','url','raw_text'])

## 2．CleanDocument → QualityBranches
同一份原文分别比较现有清洗、Gopher式规则、C4式标点过滤、结构块过滤及段落精确去重。反例分支不作为推荐配置。

In [ ]:
filtered=data.read_records(RUN/'datasets/quality_branches.jsonl').map(lambda r:r['value']).filter(lambda r:r['doc_id'].startswith(DOC))
show(filtered,columns=['doc_id','title','variants','filter_diagnostics'])

## 3．CompareDuplicates
MinHash文档候选与段落候选分别查看。共享段落合并完全相同正文的来源位置，不把相似但有额外细节的段落强行删除。

In [ ]:
shared=data.read_records(RUN/'shared_passages.jsonl').map(lambda r:r['value']).filter(lambda r:not DOC or any(s['doc_id'].startswith(DOC) for s in r['sources']))
show(shared,columns=['text','sources'])

## 4．完整审查页面
可展开逐块决定、原文对照及两种HTML抽取器输出。新抓取HTML与历史Markdown分开比较。

In [ ]:
display(HTML((RUN/'preview.html').read_text()))